In [1]:
import pandas as pd
import numpy as np
from neo4j import GraphDatabase
from collections import Counter

## Đọc và lưu trữ thông tin cho các node

In [2]:
dt = pd.read_csv("./relation_new.csv")
data = pd.DataFrame(dt)

loc_dt = pd.read_csv('location.csv')
location_data = pd.DataFrame(loc_dt, columns=['commune', 'district','province'])

In [3]:
# Node
start_node = list(data['source'])
end_node = list(data['target'])

# Node properties
age_group = list(data['age_group'])
onset_date = list(data['onset_date'])
quarantine_date = list(data['quarantine_date'])
announce_date = list(data['announce_date'])
name = list(data['name'])

# Edge type <-> relationship type
rel_id = list(data['r_type'])
commune = list(data['commune'])
district = list(data['district'])
province = list(data['province'])


# Location info
l_commune = list(location_data['commune'])
l_district = list(location_data['district'])
l_province = list(location_data['province'])

## Cập nhật database, bao gồm việc sinh query

In [4]:
def parse(data):
    return '"{}"'.format(data)

def gen_properties(**kwargs):
    attrs = ["{}:{}".format(k, v) for k, v in kwargs.items() ]
    query = ", ".join(attrs)
    query = "{"+query+"}"
    return query

def gen_query_node(label,**kwargs):
    attr = gen_properties(**kwargs)
    query = ("""
    CREATE (n:{} {})
    """).format(label, attr)
    return query

In [5]:
uri = 'bolt://localhost:11005'

g = GraphDatabase.driver(uri, auth=('neo4j', '123'))

In [6]:
q = "MATCH (n) DETACH DELETE n"

g.session().run(q)

In [7]:
def import_db():
    with g.session() as session:
        for i in range(len(end_node)):
            try:
                session.run(gen_query_node('Patient', name=parse(end_node[i]), full_name=parse(name[i]),quarantine_date = 'date({})'.format(parse(quarantine_date[i])) ,onset_date='date({})'.format(parse(onset_date[i])), announce_date='date({})'.format(parse(announce_date[i])), age_group=age_group[i], commune=parse(commune[i]), district=parse(district[i]),province=parse(province[i])))
            except:
                pass

import_db()

In [8]:
rel_type = {'0': "UNKNOWN", '1': "STAFF_PATIENT", '2': "FELLOW", '3':"RELATIVES", '4':"SOCIAL"}
def check_exist_edge(s, t):
    q = ("""
    MATCH (s:Patient)-[r]->(t:Patient)
    WHERE s.name="{}" AND t.name="{}"
    RETURN r
    """).format(s, t)
    with g.session() as session:
        return session.run(q).values()


def gen_query_edge(src, target, rel_id):
    r_type = rel_type[str(rel_id)]
    query = ("""
    MATCH (s:Patient), (t:Patient)
    WHERE s.name = "{0}" AND t.name = "{1}"
    CREATE (s)-[r:{2}]->(t)
    """).format(src, target, r_type)
    return query

In [9]:
def create_relationship(start_node, end_node, rel_id):
    with g.session() as session:
        for s, t, r in zip(start_node, end_node, rel_id):
            if s!='0' and len(check_exist_edge(t, s))==0:
                session.run(gen_query_edge(t, s, r))
                              
                              
create_relationship(start_node, end_node, rel_id)

## Add locations

In [10]:
def add_location(commune, district, province, type):
    with g.session() as session:
        for c, d, p in zip(commune, district, province):
            if type=='commune':
                try:
                    session.run(gen_query_node('Location', name=parse(c), l_type=parse('commune'), commune = parse(c), district = parse(d), province = parse(p)))
                except:
                    pass
            elif type=='district':
                try:
                    session.run(gen_query_node('Location', name=parse(d), l_type=parse('district'), commune='"None"', district = parse(d), province = parse(p)))
                except:
                    pass
            elif type == 'province':
                try:
                    session.run(gen_query_node('Location', name = parse(p), l_type=parse('province'), commune='"None"', district='"None"', province=parse(p)))
                except:
                    pass
                    
add_location(l_commune, l_district, l_province, 'commune')
add_location(l_commune, l_district, l_province, 'district')
add_location(l_commune, l_district, l_province, 'province')

In [11]:
def gen_query_edge(source, target, type):
    p = '{name:"'+type+'"}'
    query = ("""
    MATCH (s:Location), (t:Location)
    WHERE s.name="{0}" AND t.name="{1}"
    CREATE (s)-[r:{2}{3}]->(t)
    """).format(source, target, type, p)
    return query

def check_connection(s, t):
    query = ("""
    MATCH (s:Location)-[r]-(t:Location)
    WHERE s.name="{0}" AND t.name="{1}"
    RETURN r
    """).format(s, t)
    with g.session() as session:
        r = session.run(query).values()
    return r


def add_edge(source, target, type):
    with g.session() as session:
        for i in range(len(source)):
            if len(check_connection(source[i], target[i]))==0:
                session.run(gen_query_edge(source[i], target[i], type))
           

add_edge(l_commune, l_district, "COMMUNE_DISTRICT")
add_edge(l_district, l_province, 'DISTRICT_PROVINCE')

## Match patient to location

In [12]:
def gen_query_loc():
    query = ("""
    MATCH (n:Patient), (l:Location)
    WHERE n.commune=l.commune AND n.district = l.district AND n.province=l.province AND l.l_type='commune'
    CREATE (n)-[r:LIVE_IN]->(l)
    """)
    return query

In [13]:
with g.session() as session:
    session.run(gen_query_loc())